# RSSI Data Analysis — Indoor Positioning

**Ziel:** Überblick über die BLE-RSSI-Daten gewinnen, Positionswahrscheinlichkeiten schätzen und Fehlerrate ermitteln.

**Methode:** Log-Distance Path Loss Model → Distanzschätzung → Weighted Centroid / Multilateration (Least Squares)

**Datenquelle:** `emi_nav.db` — Tabellen `runs`, `ble_rssi`

**Referenz:** PositionEstimation.pdf (Prof. Dr. Gerald Pirkl, OTH Amberg-Weiden)

## 0 — Imports & Config

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.optimize import minimize
from scipy.stats import gaussian_kde
import warnings
warnings.filterwarnings('ignore')

DB_PATH = '../data/emi_nav.db'  # Pfad zur SQLite-Datenbank

# ---------------------------------------------------------------
# Beacon-Koordinaten (Meter, Gebäude-Koordinatensystem)
# ACHTUNG: Diese Werte müssen mit den echten Positionen befüllt werden!
# Format: beacon_name -> (x_m, y_m, floor)
# floor: 0 = EG, 1 = OG
# ---------------------------------------------------------------
BEACON_POSITIONS = {
    # Beispiel-Einträge — BITTE MIT ECHTEN WERTEN ERSETZEN
    'arrive_emi1': (0.0,  0.0,  0),
    'arrive_emi2': (10.0, 0.0,  0),
    'arrive_emi3': (5.0,  8.0,  0),
    'arrive_emi4': (0.0,  8.0,  1),
    'arrive_emi5': (10.0, 8.0,  1),
    'arrive_emi6': (5.0,  0.0,  1),
}

# ---------------------------------------------------------------
# Groundtruth-Zeitpunkte pro Run
# Format: run_id -> list of (timestamp_ms, label)
# Pixel-Koordinaten noch unbekannt — nur Zeitstempel + Label
# ---------------------------------------------------------------
GROUNDTRUTH = {
    'R1': [
        # (timestamp_ms, label) — BITTE BEFÜLLEN
        # Beispiel: (1718543000000, 'Punkt A')
    ],
    'R2': [],
    'R3': [],
}

# Path Loss Modell Parameter (Log-Distance)
# RSSI(d) = RSSI_0 - 10 * n * log10(d)
RSSI_0  = -40.0   # RSSI bei d=1m (dBm) — kalibrieren!
PATH_LOSS_EXP = 2.5  # n: Freifeld ~2, Innenraum 2.5-4
FLOOR_PENALTY_DB = 15.0  # Extra Dämpfung pro Stockwerk (dB)

print('Config geladen ✓')

## 1 — Daten laden

In [ ]:
conn = sqlite3.connect(DB_PATH)

runs_df = pd.read_sql('SELECT * FROM runs', conn)
rssi_df = pd.read_sql('SELECT * FROM ble_rssi', conn)
conn.close()

print(f'Runs: {len(runs_df)}')
print(runs_df)
print(f'\nRSSI-Messungen: {len(rssi_df)}')
print(rssi_df.head())
print('\nBeacon-Namen:', rssi_df['beacon_name'].unique())

## 2 — Explorative Datenanalyse (EDA)

In [ ]:
print('=== Allgemeine Stats ===')
print(rssi_df.describe())
print('\n=== Messungen pro Run & Beacon ===')
print(rssi_df.groupby(['run_id', 'beacon_name'])['rssi'].agg(['count','mean','std','min','max']).round(2))

In [ ]:
beacons = sorted(rssi_df['beacon_name'].unique())
runs    = sorted(rssi_df['run_id'].unique())
colors  = cm.tab10(np.linspace(0, 1, len(runs)))

fig, axes = plt.subplots(len(beacons), 1, figsize=(12, 3 * len(beacons)), squeeze=False)

for i, beacon in enumerate(beacons):
    ax = axes[i, 0]
    for run, col in zip(runs, colors):
        subset = rssi_df[(rssi_df['beacon_name'] == beacon) & (rssi_df['run_id'] == run)]
        if len(subset) > 1:
            subset = subset.sort_values('timestamp_ms')
            ax.plot(subset['timestamp_ms'], subset['rssi'], label=run, color=col, alpha=0.8, linewidth=0.8)
    ax.set_title(f'RSSI über Zeit — {beacon}')
    ax.set_xlabel('Timestamp (ms)')
    ax.set_ylabel('RSSI (dBm)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('rssi_over_time.png', dpi=120)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
rssi_df.boxplot(column='rssi', by=['beacon_name', 'run_id'], ax=ax, rot=45)
ax.set_title('RSSI Verteilung pro Beacon & Run')
ax.set_xlabel('Beacon / Run')
ax.set_ylabel('RSSI (dBm)')
plt.suptitle('')
plt.tight_layout()
plt.savefig('rssi_boxplot.png', dpi=120)
plt.show()

## 3 — Stockwerk-Klassifikation

In [ ]:
eg_beacons = [b for b, (_, _, f) in BEACON_POSITIONS.items() if f == 0]
og_beacons = [b for b, (_, _, f) in BEACON_POSITIONS.items() if f == 1]

print(f'EG-Beacons ({len(eg_beacons)}): {eg_beacons}')
print(f'OG-Beacons ({len(og_beacons)}): {og_beacons}')

def estimate_floor(rssi_window: pd.DataFrame) -> int:
    """
    Heuristik: Vergleich mittlerer RSSI EG- vs OG-Beacons.
    Stärkere Signale = wahrscheinlich selbes Stockwerk.
    """
    rssi_eg = rssi_window[rssi_window['beacon_name'].isin(eg_beacons)]['rssi'].mean()
    rssi_og = rssi_window[rssi_window['beacon_name'].isin(og_beacons)]['rssi'].mean()
    if pd.isna(rssi_eg) and pd.isna(rssi_og): return -1
    if pd.isna(rssi_eg): return 1
    if pd.isna(rssi_og): return 0
    return 0 if rssi_eg >= rssi_og else 1

## 4 — Distanzschätzung via Log-Distance Path Loss

In [ ]:
def rssi_to_distance(rssi: float, floor_delta: int = 0) -> float:
    """
    Inverse Log-Distance Path Loss (PositionEstimation.pdf, S.11):
      d = 10^((RSSI_0 - RSSI - floor_penalty) / (10 * n))
    """
    adjusted_rssi = rssi - floor_delta * FLOOR_PENALTY_DB
    exp = (RSSI_0 - adjusted_rssi) / (10.0 * PATH_LOSS_EXP)
    return 10.0 ** exp

rssi_df['beacon_floor'] = rssi_df['beacon_name'].map(
    lambda b: BEACON_POSITIONS.get(b, (None, None, 0))[2]
)
rssi_df['floor_delta'] = rssi_df['beacon_floor'].fillna(0).astype(int)
rssi_df['dist_est_m']  = rssi_df.apply(
    lambda r: rssi_to_distance(r['rssi'], r['floor_delta']), axis=1
)

print('Distanzschätzungen (Beispiel):')
print(rssi_df[['run_id','beacon_name','rssi','floor_delta','dist_est_m']].head(10).round(2))

## 5 — Multilateration (Least Squares)

In [ ]:
def multilaterate_ls(beacon_positions: list, distances: list) -> tuple:
    """
    Least Squares Multilateration 2D.
    Linearisierung durch Subtraktion des ersten Ankerpunkts.
    Quelle: PositionEstimation.pdf S.24-26.
    Returns (x_est, y_est, residual)
    """
    if len(beacon_positions) < 3:
        return None, None, None
    beacons = np.array(beacon_positions)
    dists   = np.array(distances)
    x1, y1, d1 = beacons[0, 0], beacons[0, 1], dists[0]
    A, b = [], []
    for i in range(1, len(beacons)):
        xi, yi, di = beacons[i, 0], beacons[i, 1], dists[i]
        A.append([2*(xi - x1), 2*(yi - y1)])
        b.append(di**2 - d1**2 - xi**2 + x1**2 - yi**2 + y1**2)
    A, b = np.array(A), np.array(b)
    result, residuals, rank, _ = np.linalg.lstsq(A, b, rcond=None)
    res = float(residuals[0]) if len(residuals) > 0 else np.nan
    return result[0], result[1], res


def estimate_positions_for_run(run_id: str, window_ms: int = 3000) -> pd.DataFrame:
    """Schätzt Positionen in rollenden Zeitfenstern."""
    run_data = rssi_df[rssi_df['run_id'] == run_id].sort_values('timestamp_ms')
    t_start, t_end = run_data['timestamp_ms'].min(), run_data['timestamp_ms'].max()
    results, t = [], t_start
    while t < t_end:
        window = run_data[(run_data['timestamp_ms'] >= t) & (run_data['timestamp_ms'] < t + window_ms)]
        avg = window.groupby('beacon_name').agg(
            rssi=('rssi', 'mean'), floor_delta=('floor_delta', 'first')
        ).reset_index()
        avg = avg[avg['beacon_name'].isin(BEACON_POSITIONS)].copy()
        avg['dist'] = avg.apply(lambda r: rssi_to_distance(r['rssi'], r['floor_delta']), axis=1)
        est_floor = estimate_floor(window)
        positions = [(BEACON_POSITIONS[b][0], BEACON_POSITIONS[b][1]) for b in avg['beacon_name']]
        x_est, y_est, residual = multilaterate_ls(positions, avg['dist'].tolist())
        results.append({
            'run_id': run_id, 'timestamp_ms': t + window_ms // 2,
            'x_est': x_est, 'y_est': y_est, 'floor_est': est_floor,
            'n_beacons': len(avg), 'residual': residual,
        })
        t += window_ms
    return pd.DataFrame(results).dropna(subset=['x_est', 'y_est'])


all_positions = pd.concat(
    [estimate_positions_for_run(r) for r in runs_df['run_id']],
    ignore_index=True
)
print(f'Positionsschätzungen gesamt: {len(all_positions)}')
print(all_positions.head(10).round(2))

## 6 — Pfad-Visualisierung pro Run

In [ ]:
runs_list = all_positions['run_id'].unique()
fig, axes = plt.subplots(len(runs_list), 1, figsize=(9, 6 * len(runs_list)), squeeze=False)

for ax, run in zip(axes[:, 0], runs_list):
    df_run = all_positions[all_positions['run_id'] == run].sort_values('timestamp_ms')
    ax.plot(df_run['x_est'], df_run['y_est'],
            color='steelblue', linewidth=1.5, zorder=1, alpha=0.7)
    sc = ax.scatter(df_run['x_est'], df_run['y_est'],
                    c=range(len(df_run)), cmap='viridis', s=30, zorder=3)
    plt.colorbar(sc, ax=ax, label='Zeitschritt')
    ax.scatter(*df_run[['x_est','y_est']].iloc[0],
               color='green', s=120, zorder=4, label='Start', marker='D')
    ax.scatter(*df_run[['x_est','y_est']].iloc[-1],
               color='red',   s=120, zorder=4, label='Ende',  marker='X')
    for name, (bx, by, bf) in BEACON_POSITIONS.items():
        marker = '^' if bf == 0 else 'v'
        ax.scatter(bx, by, color='orange', s=80, marker=marker, zorder=5, alpha=0.8)
        ax.annotate(name, (bx, by), textcoords='offset points',
                    xytext=(5, 5), fontsize=7, color='darkorange')
    ax.set_title(f'Run {run} — Geschätzte Position (Pfad)')
    ax.set_xlabel('X [m]'); ax.set_ylabel('Y [m]')
    ax.legend(); ax.set_aspect('equal'); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('path_per_run.png', dpi=130)
plt.show()

## 7 — Positionswahrscheinlichkeit (KDE Heatmap)

In [ ]:
fig, axes = plt.subplots(1, len(runs_list), figsize=(6 * len(runs_list), 5), squeeze=False)

for ax, run in zip(axes[0], runs_list):
    df_run = all_positions[all_positions['run_id'] == run].dropna()
    if len(df_run) < 5:
        ax.set_title(f'Run {run} — zu wenig Daten'); continue
    xy  = np.vstack([df_run['x_est'], df_run['y_est']])
    kde = gaussian_kde(xy)
    x_grid = np.linspace(df_run['x_est'].min()-2, df_run['x_est'].max()+2, 100)
    y_grid = np.linspace(df_run['y_est'].min()-2, df_run['y_est'].max()+2, 100)
    X, Y = np.meshgrid(x_grid, y_grid)
    Z = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)
    im = ax.contourf(X, Y, Z, levels=20, cmap='hot_r', alpha=0.85)
    plt.colorbar(im, ax=ax, label='Wahrscheinlichkeitsdichte')
    ax.scatter(df_run['x_est'], df_run['y_est'], s=10, color='white', alpha=0.4, zorder=2)
    ax.set_title(f'Run {run} — Positionswahrscheinlichkeit (KDE)')
    ax.set_xlabel('X [m]'); ax.set_ylabel('Y [m]')
    ax.set_aspect('equal'); ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('position_probability_kde.png', dpi=130)
plt.show()

## 8 — Fehleranalyse (Groundtruth)

In [ ]:
# ---------------------------------------------------------------
# HINWEIS: Groundtruth-Pixel-Koordinaten noch unbekannt.
# Sobald Pixel → Meter Konvertierung vorliegt: GROUNDTRUTH_XY befüllen.
# Format: run_id -> list of (timestamp_ms, x_gt_m, y_gt_m, label)
# ---------------------------------------------------------------
GROUNDTRUTH_XY = {
    'R1': [
        # Beispiel: (1718543000000, 3.5, 4.2, 'Punkt A')
    ],
    'R2': [],
    'R3': [],
}


def compute_errors(run_id, pos_df, gt):
    errors = []
    for (t_gt, x_gt, y_gt, label) in gt:
        row = pos_df.iloc[(pos_df['timestamp_ms'] - t_gt).abs().argsort()[:1]]
        if len(row) == 0: continue
        x_est, y_est = row['x_est'].values[0], row['y_est'].values[0]
        err = np.sqrt((x_est - x_gt)**2 + (y_est - y_gt)**2)
        errors.append({'run_id': run_id, 'label': label,
                        'x_gt': x_gt, 'y_gt': y_gt,
                        'x_est': x_est, 'y_est': y_est, 'error_m': err})
    return pd.DataFrame(errors)


error_dfs = []
for run_id in runs_df['run_id']:
    gt = GROUNDTRUTH_XY.get(run_id, [])
    if not gt:
        print(f'Run {run_id}: Keine Groundtruth-Koordinaten hinterlegt.')
        continue
    run_pos = all_positions[all_positions['run_id'] == run_id]
    error_dfs.append(compute_errors(run_id, run_pos, gt))

if error_dfs:
    all_errors = pd.concat(error_dfs, ignore_index=True)
    print('=== Fehlerstatistik (Meter) ===')
    print(all_errors.groupby('run_id')['error_m'].agg(['mean','median','std','max']).round(2))
    print(f'\nGesamtfehler (RMSE): {np.sqrt((all_errors["error_m"]**2).mean()):.2f} m')
else:
    print('Fehleranalyse: GROUNDTRUTH_XY befüllen um Fehler zu berechnen.')

In [ ]:
if error_dfs and len(all_errors) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    sorted_err = np.sort(all_errors['error_m'])
    cdf = np.arange(1, len(sorted_err)+1) / len(sorted_err)
    axes[0].plot(sorted_err, cdf, color='steelblue', linewidth=2)
    axes[0].axvline(np.median(sorted_err), color='red', linestyle='--',
                    label=f'Median: {np.median(sorted_err):.2f} m')
    axes[0].set_xlabel('Positionsfehler (m)'); axes[0].set_ylabel('CDF')
    axes[0].set_title('Kumulierte Fehlerverteilung (CDF)')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)
    all_errors.boxplot(column='error_m', by='run_id', ax=axes[1])
    axes[1].set_title('Fehler pro Run'); axes[1].set_xlabel('Run')
    axes[1].set_ylabel('Fehler (m)'); plt.suptitle('')
    plt.tight_layout()
    plt.savefig('error_analysis.png', dpi=130)
    plt.show()
else:
    print('Fehler-Plot: Groundtruth-Daten erforderlich.')

## 9 — Zusammenfassung & Nächste Schritte

### Was wurde gemacht
1. **EDA** — RSSI-Verteilung pro Beacon & Run analysiert
2. **Log-Distance Path Loss** → Distanzschätzung aus RSSI (mit Stockwerk-Penalty)
3. **Multilateration (Least Squares)** → x/y-Positionsschätzung in rollenden Fenstern
4. **KDE Heatmap** → Positionswahrscheinlichkeit visualisiert
5. **Fehleranalyse** → Vorbereitet, wartet auf Groundtruth-Koordinaten

### TODOs
- [ ] `BEACON_POSITIONS` mit echten Meterwerten befüllen
- [ ] Groundtruth Pixel → Meter konvertieren und in `GROUNDTRUTH_XY` eintragen
- [ ] `RSSI_0` & `PATH_LOSS_EXP` kalibrieren (z.B. mit bekannten Testpunkten)
- [ ] `floor_delta` dynamisch pro Zeitpunkt schätzen statt statisch
- [ ] Fingerprinting als Alternative evaluieren (falls Referenzdaten vorhanden)